In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# WEEK 7 — FUNCTION 4 (REVISED v3)
# Required changes implemented (based on your latest output):
#  1) Still tunes hyperparameters via Random Search + K-fold CV
#  2) Still uses bootstrapped ensemble for uncertainty
#  3) Prints TOP-K non-duplicate candidates (mean/std/EI/PI)
#  4) NEW selection policy (to increase chance of beating best):
#        "Hybrid PI-first among near-best EI"
#        - Compute EI for all candidates
#        - Keep candidates with EI >= (1 - ei_band) * max_EI
#        - Choose highest PI among that band
#        - Tie-break: higher mean, then higher EI
#     (This avoids picking a point whose EI is only high because sigma is huge.)
# ============================================================

# ============================================================
# 1) Input data (Function 4)
# ============================================================
X_train = np.array([
    [0.89698105, 0.72562797, 0.17540431, 0.70169437],
    [0.8893564 , 0.49958786, 0.53926886, 0.50878344],
    [0.25094624, 0.03369313, 0.14538002, 0.49493242],
    [0.34696206, 0.0062504 , 0.76056361, 0.61302356],
    [0.12487118, 0.12977019, 0.38440048, 0.2870761 ],
    [0.80130271, 0.50023109, 0.70664456, 0.19510284],
    [0.24770826, 0.06044543, 0.04218635, 0.44132425],
    [0.74670224, 0.7570915 , 0.36935306, 0.20656628],
    [0.40066503, 0.07257425, 0.88676825, 0.24384229],
    [0.6260706 , 0.58675126, 0.43880578, 0.77885769],
    [0.95713529, 0.59764438, 0.76611385, 0.77620991],
    [0.73281243, 0.14524998, 0.47681272, 0.13336573],
    [0.65511548, 0.07239183, 0.68715175, 0.08151656],
    [0.21973443, 0.83203134, 0.48286416, 0.08256923],
    [0.48859419, 0.2119651 , 0.93917791, 0.37619173],
    [0.16713049, 0.87655456, 0.21723954, 0.95980098],
    [0.21691119, 0.16608583, 0.24137226, 0.77006248],
    [0.38748784, 0.80453226, 0.75179548, 0.72382744],
    [0.98562189, 0.66693268, 0.15678328, 0.8565348 ],
    [0.03782483, 0.66485335, 0.16198218, 0.25392378],
    [0.68348638, 0.9027701 , 0.33541983, 0.99948256],
    [0.17034731, 0.75695908, 0.27652049, 0.5312315 ],
    [0.85965692, 0.91959232, 0.20613873, 0.09779683],
    [0.28213837, 0.50598691, 0.53053084, 0.09630162],
    [0.32607578, 0.4723669 , 0.453192  , 0.10588734],
    [0.94838936, 0.89451301, 0.85163782, 0.55219629],
    [0.66495539, 0.04656628, 0.11677747, 0.79371778],
    [0.57776561, 0.42877174, 0.42582587, 0.24900741],
    [0.73861301, 0.48210263, 0.70936644, 0.50397001],
    [0.8548108 , 0.49396462, 0.73530997, 0.80809201],
    [1.085621  , 1.019592  , 1.039177  , 1.099482  ],
    [1.00000e-06, 1.00000e-06, 1.24558e-01, 1.00000e-06],
    [0.866175  , 0.601115  , 0.708072  , 0.020585  ],
    [0.145904  , 0.536548  , 0.6014    , 0.01905   ],
    [0.356293  , 0.442523  , 0.13052   , 0.242559  ],
    [0.061431  , 0.381247  , 0.983792  , 0.705575  ]
], dtype=float)

y_train = np.array([
    -22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
    -15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
    -27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
    -26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
    -29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
    -32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284 ,
    -67.60493430274798, -22.782193418373407, -22.194212794446454,
    -13.363105653346768,  -5.926020577803715, -23.786955955997737
], dtype=float)

# ============================================================
# 2) Current best (maximisation)
# ============================================================
current_best_idx = int(np.argmax(y_train))
current_best_x = X_train[current_best_idx]
current_best_y = float(y_train[current_best_idx])

print("Current best index:", current_best_idx)
print("Current best X:", current_best_x)
print("Current best y:", current_best_y)

# ============================================================
# 3) Scale X to [0,1] per feature + standardise y
# ============================================================
X_min = X_train.min(axis=0)
X_max = X_train.max(axis=0)
X_scaled = (X_train - X_min) / (X_max - X_min + 1e-12)

y_mean = y_train.mean()
y_std = y_train.std() + 1e-12
y_scaled = (y_train - y_mean) / y_std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor_all = torch.tensor(y_scaled, dtype=torch.float32, device=device).unsqueeze(-1)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
rng = np.random.default_rng(42)

# ============================================================
# 4) MLP surrogate
# ============================================================
class MLP(nn.Module):
    def __init__(self, input_dim=4, hidden=(64, 64), p_dropout=0.1):
        super().__init__()
        h1, h2 = hidden
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h2, 1),
        )

    def forward(self, x):
        return self.net(x)

def train_model(model, X, y, epochs=800, lr=1e-3, weight_decay=1e-4, verbose=False):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.train()
    for ep in range(epochs):
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        if verbose and (ep + 1) % max(1, epochs // 4) == 0:
            print(f"  epoch {ep+1}/{epochs} mse={loss.item():.4f}")
    return float(loss.item())

# ============================================================
# 5) CV + Random Search tuning
# ============================================================
def kfold_indices(n, k=4, seed=123):
    rr = np.random.default_rng(seed)
    idx = np.arange(n)
    rr.shuffle(idx)
    return np.array_split(idx, k)

def cv_mse_for_config(cfg, X_tensor, y_tensor, k=4, seed=123):
    folds = kfold_indices(len(X_tensor), k=k, seed=seed)
    mses = []
    for i in range(k):
        val_idx = folds[i]
        tr_idx = np.concatenate([folds[j] for j in range(k) if j != i])

        X_tr, y_tr = X_tensor[tr_idx], y_tensor[tr_idx]
        X_val, y_val = X_tensor[val_idx], y_tensor[val_idx]

        torch.manual_seed(1000 + i)
        model = MLP(input_dim=4, hidden=cfg["hidden"], p_dropout=cfg["dropout"]).to(device)
        train_model(
            model, X_tr, y_tr,
            epochs=cfg["epochs"],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            verbose=False
        )

        model.eval()
        with torch.no_grad():
            pred = model(X_val)
            mse = nn.MSELoss()(pred, y_val).item()
        mses.append(mse)

    return float(np.mean(mses))

def sample_config(rr):
    # Stable ranges for small tabular data
    hidden_choices = [(32, 32), (64, 32), (64, 64), (128, 64)]
    dropout_choices = [0.0, 0.05, 0.1, 0.2]
    lr_choices = [3e-4, 1e-3, 3e-3]  # avoids overly-high 0.01
    wd_choices = [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
    epochs_choices = [500, 800, 1200]
    n_ens_choices = [5, 7]
    xi_choices = [0.0, 0.01, 0.02]   # exploit-leaning

    return {
        "hidden": hidden_choices[rr.integers(0, len(hidden_choices))],
        "dropout": float(dropout_choices[rr.integers(0, len(dropout_choices))]),
        "lr": float(lr_choices[rr.integers(0, len(lr_choices))]),
        "weight_decay": float(wd_choices[rr.integers(0, len(wd_choices))]),
        "epochs": int(epochs_choices[rr.integers(0, len(epochs_choices))]),
        "n_ensemble": int(n_ens_choices[rr.integers(0, len(n_ens_choices))]),
        "xi": float(xi_choices[rr.integers(0, len(xi_choices))]),
    }

N_TRIALS = 35
K_FOLDS = 4

best_cfg = None
best_cv = float("inf")

print("\n=== Hyperparameter tuning (random search + {}-fold CV) ===".format(K_FOLDS))
for t in range(N_TRIALS):
    cfg = sample_config(rng)
    cv = cv_mse_for_config(cfg, X_tensor_all, y_tensor_all, k=K_FOLDS, seed=123)

    if cv < best_cv:
        best_cv = cv
        best_cfg = cfg

    print(f"trial {t+1:02d}/{N_TRIALS}  cv_mse={cv:.6f}  cfg={cfg}")

print("\n=== Best tuned configuration ===")
print("Best CV-MSE (scaled y):", best_cv)
print("Best cfg:", best_cfg)

# ============================================================
# 6) Train tuned BOOTSTRAPPED ensemble
# ============================================================
def train_ensemble(cfg, X_tensor, y_tensor, rr):
    ensemble = []
    n = len(X_tensor)
    for m in range(cfg["n_ensemble"]):
        boot_idx = rr.integers(0, n, size=n)
        Xb = X_tensor[boot_idx]
        yb = y_tensor[boot_idx]

        torch.manual_seed(500 + m)
        model = MLP(input_dim=4, hidden=cfg["hidden"], p_dropout=cfg["dropout"]).to(device)
        train_model(
            model, Xb, yb,
            epochs=cfg["epochs"],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            verbose=False
        )
        ensemble.append(model)
    return ensemble

ensemble = train_ensemble(best_cfg, X_tensor_all, y_tensor_all, rng)

# ============================================================
# 7) Prediction (original y units) + EI/PI
# ============================================================
def ensemble_predict(ensemble, X_scaled_tensor):
    preds = []
    with torch.no_grad():
        for model in ensemble:
            model.eval()
            p_scaled = model(X_scaled_tensor).squeeze(-1)
            p_raw = p_scaled * y_std + y_mean
            preds.append(p_raw)
    preds = torch.stack(preds, dim=0)
    return preds.mean(dim=0), preds.std(dim=0) + 1e-9

normal = torch.distributions.Normal(
    torch.tensor(0.0, device=device),
    torch.tensor(1.0, device=device)
)

def expected_improvement(mean, std, best_y, xi=0.01):
    imp = mean - best_y - xi
    Z = imp / std
    ei = imp * normal.cdf(Z) + std * torch.exp(normal.log_prob(Z))
    return torch.where(std > 0, ei, torch.zeros_like(ei))

def probability_of_improvement(mean, std, best_y, xi=0.0):
    imp = mean - best_y - xi
    Z = imp / std
    return normal.cdf(Z)

# ============================================================
# 8) Candidate search + NEW selection policy
# ============================================================
def is_duplicate(x, X_existing, tol=1e-6):
    return np.any(np.linalg.norm(X_existing - x, axis=1) < tol)

def propose_next_point_hybrid(
    ensemble, X_min, X_max, X_existing,
    rr,
    n_candidates=40000,
    xi=0.01,
    top_k=10,
    dup_tol=1e-6,
    ei_band=0.05,      # keep candidates within 5% of max EI
):
    """
    Hybrid selection:
      1) Compute EI/PI for many candidates
      2) Remove duplicates
      3) Find max_EI among remaining
      4) Keep candidates with EI >= (1 - ei_band) * max_EI
      5) Choose the one with highest PI (tie-break: higher mean, then higher EI)
    """
    candidates_scaled = rr.random((n_candidates, 4), dtype=np.float32)
    X_cand_tensor = torch.tensor(candidates_scaled, dtype=torch.float32, device=device)

    mean, std = ensemble_predict(ensemble, X_cand_tensor)

    best_y = float(np.max(y_train))
    ei = expected_improvement(mean, std, best_y, xi=xi)
    pi = probability_of_improvement(mean, std, best_y, xi=0.0)

    # Build list of non-duplicate candidates
    items = []
    for idx in range(n_candidates):
        x_raw = X_min + candidates_scaled[idx] * (X_max - X_min)
        if is_duplicate(x_raw, X_existing, tol=dup_tol):
            continue
        items.append({
            "idx": idx,
            "x_raw": x_raw,
            "mean": float(mean[idx].item()),
            "std": float(std[idx].item()),
            "ei": float(ei[idx].item()),
            "pi": float(pi[idx].item())
        })

    if len(items) == 0:
        # fallback: return best EI without duplicate filtering
        best_idx = int(torch.argmax(ei).item())
        x_raw = X_min + candidates_scaled[best_idx] * (X_max - X_min)
        chosen = {
            "x_raw": x_raw,
            "mean": float(mean[best_idx].item()),
            "std": float(std[best_idx].item()),
            "ei": float(ei[best_idx].item()),
            "pi": float(pi[best_idx].item())
        }
        report = [chosen]
        return chosen, report, {"max_ei": chosen["ei"], "ei_threshold": chosen["ei"]}

    # Find max EI among non-duplicates
    max_ei = max(r["ei"] for r in items)
    ei_threshold = (1.0 - ei_band) * max_ei

    # Filter to EI-band near the top
    band = [r for r in items if r["ei"] >= ei_threshold]

    # Choose highest PI within EI-band; tie-break by mean then EI
    chosen = max(band, key=lambda r: (r["pi"], r["mean"], r["ei"]))

    # For reporting: show TOP-K by EI (still useful for sanity checks)
    items_sorted_ei = sorted(items, key=lambda r: r["ei"], reverse=True)
    report = items_sorted_ei[:top_k]

    meta = {"max_ei": max_ei, "ei_threshold": ei_threshold}
    return chosen, report, meta

chosen, report, meta = propose_next_point_hybrid(
    ensemble,
    X_min, X_max,
    X_existing=X_train,
    rr=rng,
    n_candidates=40000,
    xi=best_cfg["xi"],
    top_k=10,
    dup_tol=1e-6,
    ei_band=0.05  # <-- tighten to 0.03 to be even more EI-faithful; loosen to 0.10 to prioritise PI more
)

next_x = chosen["x_raw"]
next_mean = chosen["mean"]
next_std = chosen["std"]
next_ei = chosen["ei"]
next_pi = chosen["pi"]

# ============================================================
# 9) Report
# ============================================================
print("\n================ WEEK 7 FUNCTION 4 RESULTS (REVISED v3) ================")
print("Tuned surrogate config:", best_cfg)
print("Best CV-MSE (scaled y):", best_cv)

print("\nCURRENT BEST OBSERVED")
print("x_best =", current_best_x, ", y_best =", current_best_y)

print("\nSELECTION POLICY")
print(f"- Hybrid: choose highest PI among candidates within {int(100*0.05)}% of max EI")
print(f"- max_EI (non-duplicate candidates) = {meta['max_ei']:.6f}")
print(f"- EI threshold for PI-selection    = {meta['ei_threshold']:.6f}")

print("\nTOP-10 NON-DUPLICATE CANDIDATES (ranked by EI)")
for i, r in enumerate(report, 1):
    x = r["x_raw"]
    print(f"{i:02d}) x={x} | mean={r['mean']:.6f} std={r['std']:.6f} EI={r['ei']:.6f} PI={r['pi']:.6f}")

print("\nRECOMMENDED NEXT POINT (Hybrid: PI-first within EI-band)")
print("x_next    =", next_x)
print("mu(x_next)=", next_mean)
print("sigma     =", next_std)
print("EI        =", next_ei)
print("PI        =", next_pi)

improvement_margin = next_mean - current_best_y
print("\nREASONING")
print(f"- Surrogate tuned by random search + {K_FOLDS}-fold CV (better CV-MSE => more reliable mean).")
print(f"- Acquisition uses xi={best_cfg['xi']} (smaller xi = more exploitative).")
print(f"- Hybrid rule avoids picking points where EI is high purely due to very large sigma.")
print(f"- Predicted mean at x_next is {next_mean:.3f} vs best {current_best_y:.3f} (Δ={improvement_margin:.3f}).")
print(f"- Uncertainty sigma={next_std:.3f} => EI={next_ei:.3f}, PI={next_pi:.3f} (~{next_pi*100:.1f}% chance to beat best).")
print("- Duplicate avoidance prevents wasting a query on an already-tested point.")


Current best index: 27
Current best X: [0.57776561 0.42877174 0.42582587 0.24900741]
Current best y: -4.02554228

=== Hyperparameter tuning (random search + 4-fold CV) ===
trial 01/35  cv_mse=0.496266  cfg={'hidden': (32, 32), 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 1e-05, 'epochs': 800, 'n_ensemble': 7, 'xi': 0.0}
trial 02/35  cv_mse=0.259878  cfg={'hidden': (64, 64), 'dropout': 0.0, 'lr': 0.0003, 'weight_decay': 1e-05, 'epochs': 1200, 'n_ensemble': 7, 'xi': 0.02}
trial 03/35  cv_mse=0.353610  cfg={'hidden': (64, 64), 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0, 'epochs': 1200, 'n_ensemble': 5, 'xi': 0.01}
trial 04/35  cv_mse=0.191252  cfg={'hidden': (64, 32), 'dropout': 0.0, 'lr': 0.003, 'weight_decay': 0.0001, 'epochs': 800, 'n_ensemble': 5, 'xi': 0.02}
trial 05/35  cv_mse=0.374151  cfg={'hidden': (64, 64), 'dropout': 0.05, 'lr': 0.001, 'weight_decay': 1e-06, 'epochs': 500, 'n_ensemble': 7, 'xi': 0.02}
trial 06/35  cv_mse=0.416194  cfg={'hidden': (32, 32), 'dropout': 0.2, 